In [1]:
import pandas as pd
DORI_ONC = pd.read_csv("/data/DORI/DORI-ONC/DORI.csv")
from pathlib import Path
import re

csv_files = DORI_ONC["filename"].apply(lambda f: Path(f).name)


DORI_ROOT = Path("/data/DORI")
DATA_DIRS = [
    DORI_ROOT / "DORI-ONC",
    DORI_ROOT / "DORI-OOI",
    DORI_ROOT / "DORI-Orcasound",
    DORI_ROOT / "DORI-SanctSound",
    DORI_ROOT / "ONC-Benchmark"
]

files_dict = {d:[] for d in DATA_DIRS}
for d in DATA_DIRS:
    repo_files = {
        p.name
        for p in d.rglob("*")
        if p.is_file() and p.suffix.lower() in (".flac", ".wav", ".mp3")
    }
    for p in  d.rglob("*"):
        if p.is_file():
            if p.suffix.lower() in (".flac", ".wav", ".mp3"):
                files_dict[d].append(p)
existing_files = []
for i, (root, files) in enumerate(files_dict.items()):
    existing_files +=[f.name for f in files]

csv_files = set(csv_files)
existing_files = set(existing_files)
print("total on csv missing locally:", len(csv_files - existing_files))
print("total local not in the CSV:", len(existing_files - csv_files))
print("overlapping files:", len(csv_files.intersection(existing_files)))



total on csv missing locally: 32403
total local not in the CSV: 9118
overlapping files: 26070


In [2]:
from huggingface_hub import HfApi
from collections import defaultdict
api = HfApi()
HF_ORG = "DORI-SRKW"
REPOS = ["DORI-ONC", "DORI-OOI", "DORI-Orcasound", "DORI-SanctSound", "ONC-Benchmark"]
common_file_roots = {r:[] for r in REPOS}
    
total_hf_files = []
for repo in REPOS:
    hf_files = api.list_repo_files(f"{HF_ORG}/{repo}", repo_type="dataset")
    hf_files = {Path(f).name for f in hf_files if f.lower().endswith((".flac", ".wav", ".mp3"))}
    # account for hf files
    for p in hf_files:
        m = re.match(r"^([A-Za-z-]+)", p)
        common_file_roots[repo].append(m.group(1) if m else None)
    total_hf_files += hf_files

total_hf_files = set(total_hf_files)
del common_file_roots['ONC-Benchmark']
common_file_roots = {k:set(v) for k, v in common_file_roots.items()}
# prefixes that appear in the CSV but in no HF repo: assign by repo name
known_roots = set().union(*common_file_roots.values())
for p in csv_files:
    m = re.match(r"^([A-Za-z-]+)", str(p))
    if m and m.group(1) not in known_roots:
        for repo in common_file_roots:
            if m.group(1).lower() in repo.lower():
                common_file_roots[repo].add(m.group(1))

print("file roots from deployments (both hf api and DORI.csv):")
for k, v in common_file_roots.items():
    print(k, ":", v)
    
missing_files_csv = csv_files - total_hf_files
missing_files_hf = total_hf_files - csv_files

print("total files on hf api:", len(total_hf_files))

print("total in csv missing in hf_api:", len(missing_files_csv))
missing_file_counts_csv = defaultdict(lambda: 0)
unresolved_files = []
for m in missing_files_csv:
    for k, v in common_file_roots.items():
        if any(v_i in m for v_i in v):
            missing_file_counts_csv[k] += 1
            break
    else:
        missing_file_counts_csv["unresolved"]+=1
        unresolved_files.append(m)
for k, v in missing_file_counts_csv.items():
    print(f"{k}: {v}")

pd.DataFrame({"filename": list(missing_files_csv)}).to_csv("./files_in_csv_not_hf_api.csv")
# files in api not in csv, assumed to be files without detections on them.
# print("total in hf_api not in the CSV:", len(missing_files_hf))
# missing_file_counts_hf = defaultdict(lambda: 0)
# unresolved_files = []
# for m in missing_files_hf:
#     for k, v in common_file_roots.items():
#         if any(v_i in m for v_i in v):
#             missing_file_counts_hf[k] += 1
#             break
#     else:
#         missing_file_counts_hf["unresolved"]+=1
#         unresolved_files.append(m)
# for k, v in missing_file_counts_hf.items():
#     print(f"{k}: {v}")
    
# print("overlapping files:", len(csv_files.intersection(total_hf_files)))


file roots from deployments (both hf api and DORI.csv):
DORI-ONC : {'ICLISTENHF', 'JASCOAMARHYDROPHONEB', 'IOS', 'JASCOAMARHYDROPHONEE', 'JASCOAMARHYDROPHONED', 'ICLISTENAF', 'ICHYDROPHONE', 'JASCOAMARHYDROPHONEC', 'NAXYS', 'ICLISTENLF', 'JASCOAMARHYDROPHONE'}
DORI-OOI : {'OO-HYEA'}
DORI-Orcasound : {'rpi-bush-point', 'rpi-orcasound-lab', 'OS', 'rpi-port-townsend'}
DORI-SanctSound : {'Sanct', 'SanctSound'}
total files on hf api: 35188
total in csv missing in hf_api: 32403
DORI-SanctSound: 31480
DORI-OOI: 491
unresolved: 369
DORI-ONC: 63
